# Fetch OpenAlex Metadata (2022–2025)

**Purpose:** Fetch full metadata (abstract, referenced_works, cited_by_count) for all new 2022-2025 papers directly by OpenAlex ID.

- Metadata JSONs contain `title`, `year`, `cited_by_count` but **no abstract**
- Abstracts are required for SciBERT semantic novelty scoring
- `referenced_works` are needed for citation edges

**Output:** `openalex_metadata_2022_2025.csv`

In [1]:
import pandas as pd
import requests
import time
import json
from tqdm import tqdm

In [1]:
# Paths
OUT_DIR = "../outputs/final/"
INT_DIR = "../outputs/intermediate/"

# OpenAlex polite pool — add your email for higher rate limits
OPENALEX_EMAIL = "e23cseu1429@bennett.edu.in"  # ← replace with your email
BASE_URL       = "https://api.openalex.org/works"

print("Setup complete.")

Setup complete.


In [3]:
# Load new paper nodes to get OpenAlex IDs
new_pn = pd.read_csv(OUT_DIR + 'paper_nodes_2022_2025.csv')

openalex_ids = new_pn['openalex_id'].dropna().unique().tolist()
print(f"Total unique OpenAlex IDs to fetch: {len(openalex_ids)}")
print("Sample:", openalex_ids[:5])

Total unique OpenAlex IDs to fetch: 1563
Sample: ['W4309674289', 'W4224436908', 'W4384918448', 'W1994546180', 'W4366420437']


In [4]:
def reconstruct_abstract(abstract_inverted_index):
    if not abstract_inverted_index:
        return None
    word_positions = []
    for word, positions in abstract_inverted_index.items():
        for pos in positions:
            word_positions.append((pos, word))
    word_positions.sort(key=lambda x: x[0])
    return ' '.join(w for _, w in word_positions)


def fetch_batch(ids_batch, email):
    """Fetch up to 50 works in one API call using the filter endpoint."""
    ids_str = '|'.join(ids_batch)
    params  = {
        'filter'     : f'openalex_id:{ids_str}',
        'per_page'   : 50,
        'mailto'     : email,
        'select'     : 'id,title,abstract_inverted_index,cited_by_count,referenced_works,publication_year'
    }
    resp = requests.get(BASE_URL, params=params, timeout=30)
    if resp.status_code == 200:
        return resp.json().get('results', [])
    else:
        print(f" API error {resp.status_code} for batch")
        return []

print("Functions defined.")

Functions defined.


In [5]:
# Fetch in batches of 50
BATCH_SIZE = 50
results    = []
failed_ids = []

batches = [openalex_ids[i:i+BATCH_SIZE] for i in range(0, len(openalex_ids), BATCH_SIZE)]
print(f"Fetching {len(openalex_ids)} papers in {len(batches)} batches of {BATCH_SIZE}...")

for i, batch in enumerate(tqdm(batches)):
    works = fetch_batch(batch, OPENALEX_EMAIL)

    fetched_ids = set()
    for work in works:
        openalex_id = work['id'].split('/')[-1]
        fetched_ids.add(openalex_id)
        abstract = reconstruct_abstract(work.get('abstract_inverted_index'))
        results.append({
            'openalex_id'       : openalex_id,
            'year'              : work.get('publication_year'),
            'cited_by_count'    : work.get('cited_by_count', 0),
            'referenced_works'  : json.dumps(work.get('referenced_works', [])),
            'abstract'          : abstract,
        })

    # Track any IDs the API didn't return
    for oid in batch:
        if oid not in fetched_ids:
            failed_ids.append(oid)

    time.sleep(0.1)  # polite delay

print(f"\nFetched:  {len(results)}")
print(f"Failed:   {len(failed_ids)}")
if failed_ids:
    print("Failed IDs:", failed_ids[:10])

Fetching 1563 papers in 32 batches of 50...


100%|██████████| 32/32 [01:13<00:00,  2.31s/it]


Fetched:  1563
Failed:   0


In [6]:
meta_df = pd.DataFrame(results)

print("Shape:", meta_df.shape)
print("Abstract null %:", meta_df['abstract'].isna().mean().round(3))
print("Year range:", meta_df['year'].min(), '–', meta_df['year'].max())
print()
print(meta_df.head(3)[['openalex_id','year','cited_by_count','abstract']].to_string())

Shape: (1563, 5)
Abstract null %: 0.275
Year range: 2021 – 2025

   openalex_id  year  cited_by_count                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [7]:
# Add global_paper_id by merging with paper_nodes
meta_df = meta_df.merge(
    new_pn[['node_id', 'openalex_id']].rename(columns={'node_id': 'global_paper_id'}),
    on='openalex_id',
    how='left'
)

# Reorder to match old openalex_metadata_full.csv format
meta_df = meta_df[['global_paper_id', 'openalex_id', 'year',
                   'cited_by_count', 'referenced_works', 'abstract']]

meta_df['match_score'] = 1.0  # direct ID fetch — perfect match

print("Final shape:", meta_df.shape)
print("Missing global_paper_id:", meta_df['global_paper_id'].isna().sum())
print()
print(meta_df.head(3).to_string())

Final shape: (2331, 7)
Missing global_paper_id: 0

        global_paper_id  openalex_id  year  cited_by_count                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [8]:
meta_df.to_csv(INT_DIR + 'openalex_metadata_2022_2025.csv', index=False)
print(f"Saved: openalex_metadata_2022_2025.csv  ({len(meta_df)} papers)")
print(f"Papers with abstracts: {meta_df['abstract'].notna().sum()}")
print(f"Papers without abstracts: {meta_df['abstract'].isna().sum()}")

Saved: openalex_metadata_2022_2025.csv  (2331 papers)
Papers with abstracts: 1731
Papers without abstracts: 600
